# Olist Brazilian e-commerce walkthrough
A guided, reproducible tour of the project data model and headline findings.

## Setup
Download the Kaggle data with `python ../scripts/download_data.py` before running the notebook.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
RAW = ROOT / 'data' / 'raw'


## Order-grain foundation
Items, payments, and reviews are aggregated before joining so that many-to-many relationships do not inflate totals.

In [ ]:
orders = pd.read_csv(RAW / 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'])
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')
items = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')
item_orders = items.groupby('order_id', as_index=False).agg(item_revenue=('price','sum'), freight=('freight_value','sum'), items=('order_item_id','count'))
payment_orders = payments.groupby('order_id', as_index=False).payment_value.sum()
review_orders = reviews.groupby('order_id', as_index=False).review_score.mean()
facts = orders.merge(customers, on='customer_id').merge(item_orders, on='order_id', how='left').merge(payment_orders, on='order_id', how='left').merge(review_orders, on='order_id', how='left')
facts.shape


## Headline KPIs

In [ ]:
pd.Series({'orders': facts.order_id.nunique(), 'customers': facts.customer_unique_id.nunique(), 'payment_value': facts.payment_value.sum(), 'average_order_value': facts.payment_value.mean()})


## Monthly growth

In [ ]:
monthly = facts.assign(month=facts.order_purchase_timestamp.dt.to_period('M').dt.to_timestamp()).groupby('month').agg(orders=('order_id','nunique'), payment_value=('payment_value','sum'))
monthly.loc['2017-01':'2018-08'].plot(subplots=True, figsize=(12,6), title=['Orders','Payment value']);


## Delivery and reviews

In [ ]:
facts['delay_days'] = (facts.order_delivered_customer_date - facts.order_estimated_delivery_date).dt.total_seconds()/86400
facts.groupby(pd.cut(facts.delay_days, [-float('inf'),-7,0,3,7,float('inf')]), observed=True).review_score.agg(['count','mean'])


## Next steps
Run `analysis/advanced_analysis.py` for RFM segments, cohorts, seller and logistics scorecards, statistical tests, and predictive-model evaluation.